**Import Required Libraries**

In [28]:
import pandas as pd
import re
import string

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

**Load Dataset**

In [29]:
df = pd.read_csv("IMDB Dataset.csv",engine="python",on_bad_lines="skip")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


**Data Understanding**

In [30]:
print("Dataset Shape:", df.shape)

print("\nClass Distribution:")
print(df['sentiment'].value_counts())

Dataset Shape: (50000, 2)

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


**Convert Labels to Numerical**

In [31]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

**NLP Preprocessing**

In [32]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):

    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r'http\S+', '', text)

    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # remove numbers
    text = re.sub(r'\d+', '', text)

    # tokenization (fast)
    words = text.split()

    # remove stopwords
    words = [word for word in words if word not in stop_words]

    # stemming
    words = [stemmer.stem(word) for word in words]

    return " ".join(words)

**Apply Preprocessing**

In [33]:
df = df.sample(10000, random_state=42)  # reduce size for faster processing

df['clean_review'] = df['review'].apply(clean_text)

df[['review','clean_review']].head()

,review,clean_review
33553,I really liked this Summerslam due to the look...,realli like summerslam due look arena curtain ...
9427,Not many television shows appeal to quite as m...,mani televis show appeal quit mani differ kind...
199,The film quickly gets to a major chase scene w...,film quickli get major chase scene ever increa...
12447,Jane Austen would definitely approve of this o...,jane austen would definit approv onebr br gwyn...
39489,Expectations were somewhat high for me when I ...,expect somewhat high went see movi thought ste...


**Train Test Split**

In [34]:
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Feature Engineering**

Bag of Words

In [35]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

TF-IDF

In [36]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

**Model Training**

Logistic Regression

In [37]:
lr = LogisticRegression()

lr.fit(X_train_tfidf, y_train)

pred_lr = lr.predict(X_test_tfidf)

**Naive Bayes**

In [39]:
nb = MultinomialNB()

nb.fit(X_train_tfidf, y_train)

pred_nb = nb.predict(X_test_tfidf)

**Decision Tree**

In [43]:
dt = DecisionTreeClassifier()

dt.fit(X_train_tfidf, y_train)

pred_dt = dt.predict(X_test_tfidf)

**Model Evaluation**

In [44]:
print("Logistic Regression")

print("Accuracy:", accuracy_score(y_test, pred_lr))
print("Precision:", precision_score(y_test, pred_lr))
print("Recall:", recall_score(y_test, pred_lr))
print("F1 Score:", f1_score(y_test, pred_lr))

Logistic Regression
Accuracy: 0.866
Precision: 0.8500477554918816
Recall: 0.8891108891108891
F1 Score: 0.869140625


In [45]:
print("Naive Bayes")

print("Accuracy:", accuracy_score(y_test, pred_nb))
print("Precision:", precision_score(y_test, pred_nb))
print("Recall:", recall_score(y_test, pred_nb))
print("F1 Score:", f1_score(y_test, pred_nb))

Naive Bayes
Accuracy: 0.847
Precision: 0.8471528471528471
Recall: 0.8471528471528471
F1 Score: 0.8471528471528471


In [46]:
print("Decision Tree")

print("Accuracy:", accuracy_score(y_test, pred_dt))
print("Precision:", precision_score(y_test, pred_dt))
print("Recall:", recall_score(y_test, pred_dt))
print("F1 Score:", f1_score(y_test, pred_dt))


Decision Tree
Accuracy: 0.703
Precision: 0.7041123370110332
Recall: 0.7012987012987013
F1 Score: 0.7027027027027027


**Model Comparison**

In [47]:
import pandas as pd

results = pd.DataFrame({

    "Model": ["Logistic Regression","Naive Bayes","Decision Tree"],

    "Accuracy":[
        accuracy_score(y_test,pred_lr),
        accuracy_score(y_test,pred_nb),
        accuracy_score(y_test,pred_dt)
    ],

    "F1 Score":[
        f1_score(y_test,pred_lr),
        f1_score(y_test,pred_nb),
        f1_score(y_test,pred_dt)
    ]
})

results

,Model,Accuracy,F1 Score
0,Logistic Regression,0.866,0.869141
1,Naive Bayes,0.847,0.847153
2,Decision Tree,0.703,0.702703
